# Day 1 – Multimodal Assistant

**Goal:** Build a multimodal assistant that accepts voice input, generates text output, and produces speech or image responses.

**Objective:** Build a pipeline that (1) accepts voice input (or text if no audio), (2) calls an LLM for a response, (3) returns text and optionally speech or image.

**What makes this different from the previous notebook:** Notebook 02 covers each modality (text, image, speech) individually with focused examples. This notebook chains them into a single end-to-end pipeline — voice in, text processing, audio out — so you see how the pieces connect in practice.

**Prerequisites:** Completed Day 1 Modules 01 and 02. Python 3.11, .venv-day1, day-1/requirements.txt, .env (OPENAI_API_KEY), and config.json (see SETUP.md).

**Time estimate:** 30–45 minutes for the core pipeline; extensions are optional stretch goals.

## Pipeline diagram – We implement each block below

```
  ┌───────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
  │  Block 1: STT  │───▶│  Block 2:     │───▶│  Block 3:     │───▶│  Audio Reply   │
  │  Voice → Text  │     │  Chat (LLM)  │     │  TTS          │     │  (.mp3 file)  │
  │  speech_to_    │     │  chat_        │     │  text_to_     │     │               │
  │  text()        │     │  response()   │     │  speech()     │     │               │
  └───────────────┘     └──────────────┘     └──────────────┘     └──────────────┘
        ▲                       │                                         │
        │                       ▼                                         │
   Audio file (.mp3)    Optional: Block 4 (Vision)                  Play locally
   OR text fallback     image context for richer answers
```

Each section below corresponds to one block in the pipeline. The end-to-end pipeline (§5) chains all blocks together.

**Fallback path:** If you don't have an audio file, start with the text-only path (skip Block 1) and come back to audio later.

## 1. Setup – environment and configuration

**Steps:**
1. Use Python 3.11, venv `.venv-day1`, and install from `day-1/requirements.txt` (see SETUP.md).
2. Copy `.env.example` to `.env` and set `OPENAI_API_KEY`.
3. Run the setup cells below. Green messages = ready; warnings = action needed.
4. Decide: start with text-only path (fastest) or prepare a short audio file (5–10 seconds).

In [1]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv
import requests

load_dotenv(Path.cwd() / ".env")
with open(Path.cwd() / "config.json", encoding="utf-8") as f:
    CONFIG = json.load(f)

API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not API_KEY:
    print("WARNING: Set OPENAI_API_KEY in .env to run real API calls. Some cells will show structure only.")
else:
    print("API key loaded successfully.")

base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
CHAT_URL = f"{base}/chat/completions"
STT_URL = f"{base}/audio/transcriptions"
TTS_URL = f"{base}/audio/speech"
MODEL_CHAT = CONFIG.get("model_chat", "gpt-4o-mini")
MODEL_STT = CONFIG.get("model_stt", "whisper-1")
MODEL_TTS = CONFIG.get("model_tts", "tts-1")

API key loaded successfully.


### Environment validation

This cell checks that all prerequisites are in place before you start coding.

In [2]:
import sys

def validate_environment() -> list[str]:
    """Check all prerequisites. Returns list of issues (empty = all good)."""
    issues = []
    # Python version
    if sys.version_info < (3, 10):
        issues.append(f"Python 3.10+ required (you have {sys.version_info.major}.{sys.version_info.minor})")
    # API key format check
    if not API_KEY:
        issues.append("OPENAI_API_KEY not set in .env")
    elif not API_KEY.startswith("sk-"):
        issues.append(f"OPENAI_API_KEY format looks unusual (expected 'sk-...')")
    # Config check
    if not CONFIG.get("openai_api_base"):
        issues.append("config.json missing 'openai_api_base'")
    if not CONFIG.get("model_chat"):
        issues.append("config.json missing 'model_chat'")
    # Dependencies
    for mod_name in ["requests", "dotenv"]:
        try:
            __import__(mod_name)
        except ImportError:
            issues.append(f"Missing dependency: {mod_name} (pip install -r requirements.txt)")
    return issues

issues = validate_environment()
if issues:
    print("Issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("All checks passed:")
    print(f"  Python: {sys.version_info.major}.{sys.version_info.minor}")
    print(f"  API base: {CONFIG.get('openai_api_base')}")
    print(f"  Chat model: {MODEL_CHAT}")
    print(f"  STT model: {MODEL_STT}")
    print(f"  TTS model: {MODEL_TTS}")
    print(f"  API key: {'set' if API_KEY else 'NOT SET'}")

All checks passed:
  Python: 3.11
  API base: https://api.openai.com/v1
  Chat model: gpt-4o-mini
  STT model: whisper-1
  TTS model: tts-1
  API key: set


## 2. Block 1 – Voice to text (STT)

**This cell implements Block 1:** Send audio to STT API; return transcript.

**How it works:**
1. Open audio file in binary mode.
2. POST as multipart form data to the transcriptions endpoint (not JSON — audio is binary).
3. API returns JSON with a `text` field containing the transcript.

**Suggested steps for participants:**
- Start with a short test audio file (e.g. 5–10 seconds) containing a simple question.
- Run this cell with your file path and inspect the transcript.
- If you don't have audio, skip to the text-only path in Block 2 and come back later.

In [3]:
# Block 1 implementation: Speech-to-Text
SUPPORTED_AUDIO_FORMATS = {".mp3", ".mp4", ".mpeg", ".mpga", ".m4a", ".wav", ".webm"}
MAX_AUDIO_SIZE_MB = 25

def validate_audio_input(audio_path: str) -> tuple[bool, str]:
    """Validate audio file before sending to STT API."""
    path = Path(audio_path)
    if not path.exists():
        return False, f"File not found: {audio_path}"
    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_AUDIO_SIZE_MB:
        return False, f"File too large: {size_mb:.1f} MB (max {MAX_AUDIO_SIZE_MB} MB)"
    if path.suffix.lower() not in SUPPORTED_AUDIO_FORMATS:
        return False, f"Unsupported format: {path.suffix}. Supported: {SUPPORTED_AUDIO_FORMATS}"
    return True, f"Valid: {path.name} ({size_mb:.1f} MB)"

def speech_to_text(audio_path: str, api_key: str = None) -> str | None:
    """Block 1: POST audio file to STT API; return transcript."""
    api_key = api_key or API_KEY
    if not api_key:
        return None
    # Validate input
    valid, msg = validate_audio_input(audio_path)
    if not valid:
        print(f"Validation failed: {msg}")
        return None
    headers = {"Authorization": f"Bearer {api_key}"}
    with open(audio_path, "rb") as f:
        files = {"file": (os.path.basename(audio_path), f, "audio/mpeg")}
        data = {"model": MODEL_STT}
        resp = requests.post(STT_URL, files=files, data=data, headers=headers, timeout=120)
    resp.raise_for_status()
    return resp.json().get("text", "")

# Example: run with a real audio file path
# transcript = speech_to_text("question.mp3")
# print("Transcript:", transcript)
print("Block 1 defined: speech_to_text(audio_path) → transcript string")
print(f"Endpoint: {STT_URL}")
print(f"Model: {MODEL_STT}")
print(f"Supported formats: {SUPPORTED_AUDIO_FORMATS}")

Block 1 defined: speech_to_text(audio_path) → transcript string
Endpoint: https://api.openai.com/v1/audio/transcriptions
Model: whisper-1
Supported formats: {'.mpga', '.mp3', '.mpeg', '.m4a', '.webm', '.mp4', '.wav'}


### Block 1 inspection: Examine the transcript

After running STT, inspect the output: length, word count, and first 100 characters. This helps verify the transcription quality before passing to Block 2.

In [4]:
def inspect_transcript(transcript: str | None) -> dict:
    """Inspect STT output: length, word count, first 100 chars."""
    if not transcript:
        return {"status": "empty", "length": 0, "word_count": 0, "preview": ""}
    return {
        "status": "ok",
        "length": len(transcript),
        "word_count": len(transcript.split()),
        "preview": transcript[:100] + ("..." if len(transcript) > 100 else ""),
    }

# Demo with sample text (as if it came from STT)
sample_transcript = "What are the three main benefits of using REST APIs in enterprise applications?"
print("Transcript inspection:")
print(json.dumps(inspect_transcript(sample_transcript), indent=2))

Transcript inspection:
{
  "status": "ok",
  "length": 79,
  "word_count": 13,
  "preview": "What are the three main benefits of using REST APIs in enterprise applications?"
}


## 3. Block 2 – Text to LLM response (Chat)

**This cell implements Block 2:** Send user text to chat completions; return assistant text.

**How it works:**
1. Build a `messages` array with optional system prompt and user message.
2. POST as JSON to the chat/completions endpoint.
3. Parse `choices[0].message.content` from the response.

**Suggested steps for participants:**
- Try a few different user prompts (technical question, summarization request, support draft).
- Change the `system_prompt` to see how behavior changes.
- Keep examples short so you can iterate quickly during the lab.

In [5]:
# Block 2 implementation: Chat / LLM response
def chat_response(
    user_text: str,
    system_prompt: str | None = None,
    api_key: str = None,
) -> str | None:
    """Block 2: POST to chat completions; return assistant content."""
    api_key = api_key or API_KEY
    if not api_key:
        return None
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_text})
    body = {"model": MODEL_CHAT, "messages": messages, "max_tokens": 300}
    resp = requests.post(CHAT_URL, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

# Example: text in → text out
if API_KEY:
    reply = chat_response("What is the capital of France? Answer in one sentence.")
    if reply:
        print("Assistant:", reply)
else:
    print("Set OPENAI_API_KEY to run. Block 2: chat_response(user_text) -> assistant text.")

Assistant: The capital of France is Paris.


### Try different system prompts

The same user question produces different responses depending on the system prompt. Try all three and compare:

| Variant | System prompt | Expected behavior |
|---------|--------------|-------------------|
| General | Default assistant | Balanced, informative answer |
| Technical | Senior software engineer | Technical depth, code references |
| Support | Customer support agent | Friendly, actionable, step-by-step |

In [6]:
SYSTEM_PROMPTS = {
    "general": "You are a helpful assistant. Answer concisely and accurately.",
    "technical": (
        "You are a senior software engineer. Give technically precise answers. "
        "Include relevant technical details, trade-offs, and best practices."
    ),
    "support": (
        "You are a customer support agent. Be friendly, patient, and actionable. "
        "Give step-by-step instructions. Use simple language."
    ),
}

test_question = "How do I handle API rate limits in my application?"

if API_KEY:
    for variant, prompt in SYSTEM_PROMPTS.items():
        print(f"\n--- {variant.upper()} ---")
        reply = chat_response(test_question, system_prompt=prompt)
        print(reply[:250] if reply else "No response")
else:
    print("Set OPENAI_API_KEY to compare system prompt variants.")
    for variant, prompt in SYSTEM_PROMPTS.items():
        print(f"  {variant}: {prompt[:60]}...")


--- GENERAL ---
Handling API rate limits effectively involves several strategies:

1. **Read Documentation**: Understand the API's rate limit policy, including limits on requests per second/minute/hour.

2. **Error Handling**: Implement appropriate error handling fo

--- TECHNICAL ---
Handling API rate limits effectively is crucial for ensuring that your application remains reliable and performs as expected. Here are the technical details, trade-offs, and best practices for managing API rate limits:

### Understanding Rate Limits


--- SUPPORT ---
Handling API rate limits in your application is important to ensure a smooth user experience and avoid disruptions. Here’s a friendly and actionable guide to help you manage those limits effectively:

### Step 1: Understand the Rate Limits
1. **Check


### Block 2 inspection: Response metadata

Inspect the full API response to see finish_reason, token usage, and model information. This is useful for debugging and cost tracking.

In [7]:
def chat_response_with_metadata(
    user_text: str,
    system_prompt: str | None = None,
    api_key: str = None,
) -> tuple[str | None, dict]:
    """Block 2 with full response metadata for inspection."""
    api_key = api_key or API_KEY
    if not api_key:
        return None, {}
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_text})
    body = {"model": MODEL_CHAT, "messages": messages, "max_tokens": 300}
    resp = requests.post(CHAT_URL, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    metadata = {
        "model": data.get("model"),
        "finish_reason": data["choices"][0].get("finish_reason"),
        "prompt_tokens": data.get("usage", {}).get("prompt_tokens"),
        "completion_tokens": data.get("usage", {}).get("completion_tokens"),
        "total_tokens": data.get("usage", {}).get("total_tokens"),
    }
    return content, metadata

if API_KEY:
    content, meta = chat_response_with_metadata("Explain REST in one sentence.")
    print("Content:", content)
    print("Metadata:", json.dumps(meta, indent=2))
else:
    print("Set OPENAI_API_KEY to inspect response metadata.")

Content: REST (Representational State Transfer) is an architectural style for designing networked applications that uses stateless communication and standard HTTP methods to manipulate resources represented by URLs.
Metadata: {
  "model": "gpt-4o-mini-2024-07-18",
  "finish_reason": "stop",
  "prompt_tokens": 13,
  "completion_tokens": 32,
  "total_tokens": 45
}


### Block 2 extension: Multi-turn conversation

Build a short two-turn conversation to show that the model maintains context when you pass the full message history.

In [8]:
def multi_turn_chat(
    history: list[dict],
    user_text: str,
    system_prompt: str = "You are a helpful assistant.",
    api_key: str = None,
) -> tuple[str | None, list[dict]]:
    """Send multi-turn conversation. Returns (assistant_text, updated_history)."""
    api_key = api_key or API_KEY
    if not api_key:
        return None, history
    if not history or history[0].get("role") != "system":
        history = [{"role": "system", "content": system_prompt}] + history
    history.append({"role": "user", "content": user_text})
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {"model": MODEL_CHAT, "messages": history, "max_tokens": 300}
    resp = requests.post(CHAT_URL, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    reply = resp.json()["choices"][0]["message"]["content"]
    history.append({"role": "assistant", "content": reply})
    return reply, history

if API_KEY:
    h: list[dict] = []
    a1, h = multi_turn_chat(h, "What are three benefits of REST APIs?")
    print("Turn 1:", a1[:200] if a1 else "")
    a2, h = multi_turn_chat(h, "Can you elaborate on the first benefit?")
    print("\nTurn 2:", a2[:200] if a2 else "")
    print(f"\nHistory: {len(h)} messages")
else:
    print("Set OPENAI_API_KEY to test multi-turn chat.")

Turn 1: REST (Representational State Transfer) APIs offer several advantages that make them popular for web services and application development. Here are three key benefits:

1. **Simplicity and Uniform Inte

Turn 2: Certainly! The first benefit of REST APIs, which is their **simplicity and uniform interface**, can be elaborated on in several ways:

### 1. Standardized Communication

REST APIs use standard HTTP me

History: 5 messages


## 4. Block 3 – Text to speech (TTS)

**This cell implements Block 3:** Convert assistant text to audio file.

**How it works:**
1. Build a JSON body with the text, voice, and model.
2. POST to the speech endpoint.
3. Response is **binary audio** (not JSON) — save directly to file.

**Suggested steps for participants:**
- Use one of the answers from Block 2 as input to this function.
- Save the audio to a filename like `assistant_reply.mp3` and play it locally.
- Try different voices and speeds.

In [9]:
# Block 3 implementation: Text-to-Speech
TTS_VOICES = ["alloy", "echo", "fable", "onyx", "nova", "shimmer"]

def text_to_speech(
    text: str,
    output_path: str,
    voice: str = "alloy",
    speed: float = 1.0,
    api_key: str = None,
) -> bool:
    """Block 3: POST text to TTS API; save audio to file."""
    api_key = api_key or API_KEY
    if not api_key:
        return False
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {
        "model": MODEL_TTS,
        "input": text,
        "voice": voice,
        "speed": speed,
    }
    resp = requests.post(TTS_URL, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    with open(output_path, "wb") as f:
        f.write(resp.content)
    return True

print("Block 3 defined: text_to_speech(text, output_path, voice, speed)")
print(f"Endpoint: {TTS_URL}")
print(f"Model: {MODEL_TTS}")
print(f"Available voices: {TTS_VOICES}")

Block 3 defined: text_to_speech(text, output_path, voice, speed)
Endpoint: https://api.openai.com/v1/audio/speech
Model: tts-1
Available voices: ['alloy', 'echo', 'fable', 'onyx', 'nova', 'shimmer']


### Voice comparison

Generate the same text in multiple voices to hear the difference. Each file is saved separately.

In [10]:
# Voice comparison: generate same text with 3 different voices
demo_text = "Welcome to the AI assistant. How can I help you today?"
voices_to_try = ["alloy", "nova", "onyx"]

if API_KEY:
    for voice in voices_to_try:
        output_file = f"voice_demo_{voice}.mp3"
        success = text_to_speech(demo_text, output_file, voice=voice)
        if success:
            size_kb = Path(output_file).stat().st_size / 1024
            print(f"  {voice}: saved to {output_file} ({size_kb:.1f} KB)")
        else:
            print(f"  {voice}: failed")
else:
    print("Set OPENAI_API_KEY to generate voice comparison files.")
    for voice in voices_to_try:
        print(f"  Would generate: voice_demo_{voice}.mp3")

  alloy: saved to voice_demo_alloy.mp3 (60.9 KB)
  nova: saved to voice_demo_nova.mp3 (66.6 KB)
  onyx: saved to voice_demo_onyx.mp3 (65.2 KB)


### Speed variants for accessibility

Slower speed (0.8) is useful for users who need more time to process audio — elderly users, non-native speakers, or accessibility requirements.

In [ ]:
# Speed comparison: normal (1.0) vs accessible-slow (0.8)
if API_KEY:
    text_to_speech("The quarterly report shows revenue increased by twelve percent.", "speed_normal.mp3", speed=1.0)
    text_to_speech("The quarterly report shows revenue increased by twelve percent.", "speed_slow.mp3", speed=0.8)
    for f in ["speed_normal.mp3", "speed_slow.mp3"]:
        if Path(f).exists():
            print(f"  {f}: {Path(f).stat().st_size / 1024:.1f} KB")
else:
    print("Set OPENAI_API_KEY to generate speed variants.")

In [ ]:
def validate_tts_output(output_path: str) -> dict:
    """Validate TTS output file: exists, non-empty, reasonable size."""
    path = Path(output_path)
    if not path.exists():
        return {"valid": False, "reason": "File not found"}
    size_bytes = path.stat().st_size
    if size_bytes < 100:
        return {"valid": False, "reason": f"File too small ({size_bytes} bytes) — likely empty or corrupt"}
    return {
        "valid": True,
        "path": str(path),
        "size_kb": round(size_bytes / 1024, 1),
        "format": path.suffix,
    }

# Check any TTS output
for fname in ["voice_demo_alloy.mp3", "speed_normal.mp3"]:
    result = validate_tts_output(fname)
    status = "OK" if result.get("valid") else f"FAIL: {result.get('reason')}"
    print(f"  {fname}: {status}")

## Block 4 (optional): Image context

**This block implements optional image-enhanced response.** Kept minimal per course feedback (relevance). The primary flow is text + TTS; image is a stretch goal.

**Use case:** User asks about a screenshot or chart — the assistant uses both the text question and the image to give a richer answer.

In [ ]:
def chat_with_image(
    user_text: str,
    image_url: str,
    system_prompt: str = "You are a helpful assistant. Describe what you see and answer the question.",
    api_key: str = None,
) -> str | None:
    """Block 4: Chat with image context (vision). Same REST pattern, message content includes image_url."""
    api_key = api_key or API_KEY
    if not api_key:
        return None
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {
        "model": MODEL_CHAT,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": [
                {"type": "text", "text": user_text},
                {"type": "image_url", "image_url": {"url": image_url}},
            ]},
        ],
        "max_tokens": 300,
    }
    resp = requests.post(CHAT_URL, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

# Example: uncomment with a real image URL
# description = chat_with_image("What error does this screenshot show?", "https://example.com/error.png")
# print(description)
print("Block 4 defined: chat_with_image(text, image_url)")
print("Optional — use when the user provides a screenshot or chart alongside their question.")

## 5. End-to-end pipeline: Block 1 → 2 → 3

**Full pipeline:** Voice (or text fallback) → STT → Chat → TTS. This section ties all blocks together.

**Step-by-step:**
1. If audio is provided, run Block 1 (`speech_to_text`) to get `user_text`. If not, start with a text question.
2. Pass `user_text` into Block 2 (`chat_response`) to get `assistant_text`.
3. Optionally, pass `assistant_text` into Block 3 (`text_to_speech`) to get an audio reply.
4. Print both `user_text` and `assistant_text` plus timing per block.

```
  ┌──────────┐      ┌──────────┐      ┌──────────┐
  │ Block 1  │─────▶│ Block 2  │─────▶│ Block 3  │
  │ STT      │      │ Chat     │      │ TTS      │
  │ (skip if │      │ (always) │      │ (optional)│
  │ text)    │      │          │      │          │
  └──────────┘      └──────────┘      └──────────┘
```

In [ ]:
def run_assistant(
    audio_path: str | None = None,
    text_input: str | None = None,
    system_prompt: str = "You are a helpful assistant. Answer concisely.",
    tts_path: str | None = None,
    tts_voice: str = "alloy",
) -> dict:
    """Run pipeline: (Block 1) STT or text_input → (Block 2) Chat → (Block 3) optional TTS.
    Returns {user_text, assistant_text, tts_path, timings, errors}."""
    result = {
        "user_text": None,
        "assistant_text": None,
        "tts_path": None,
        "timings": {},
        "errors": [],
    }

    # Block 1: STT (or text fallback)
    if audio_path and os.path.isfile(audio_path):
        t0 = time.time()
        try:
            result["user_text"] = speech_to_text(audio_path)
        except Exception as e:
            result["errors"].append(f"STT failed: {e}")
        result["timings"]["stt_seconds"] = round(time.time() - t0, 2)
    if not result["user_text"] and text_input:
        result["user_text"] = text_input
    elif not result["user_text"]:
        result["user_text"] = "What is one key benefit of REST APIs?"
        result["errors"].append("No audio or text provided; using default question")

    # Block 2: Chat (required)
    t0 = time.time()
    try:
        result["assistant_text"] = chat_response(result["user_text"], system_prompt=system_prompt)
    except Exception as e:
        result["errors"].append(f"Chat failed: {e}")
        result["timings"]["chat_seconds"] = round(time.time() - t0, 2)
        return result
    result["timings"]["chat_seconds"] = round(time.time() - t0, 2)

    # Block 3: TTS (optional, skip on failure)
    if tts_path and API_KEY and result["assistant_text"]:
        t0 = time.time()
        try:
            text_to_speech(result["assistant_text"], tts_path, voice=tts_voice)
            result["tts_path"] = tts_path
        except Exception as e:
            result["errors"].append(f"TTS failed (text answer still available): {e}")
        result["timings"]["tts_seconds"] = round(time.time() - t0, 2)

    return result

def print_result(result: dict):
    """Pretty-print pipeline result."""
    print(f"User: {result['user_text']}")
    print(f"Assistant: {result['assistant_text']}")
    if result["tts_path"]:
        print(f"Audio saved: {result['tts_path']}")
    if result["timings"]:
        parts = [f"{k}: {v}s" for k, v in result["timings"].items()]
        print(f"Timings: {', '.join(parts)}")
    if result["errors"]:
        print(f"Warnings: {result['errors']}")

### Variant A: Text-only (no audio, no TTS)

Fastest path — type a question, get a text answer. Start here to verify Block 2 works.

In [ ]:
# Variant A: text-only
result_a = run_assistant(text_input="Explain REST in one sentence.")
print("=== Variant A: Text only ===")
print_result(result_a)

### Variant B: Text + TTS (audio output)

Same as Variant A, plus Block 3 generates an audio file of the answer.

In [ ]:
# Variant B: text + TTS
result_b = run_assistant(
    text_input="Give three benefits of multimodal AI assistants.",
    tts_path="assistant_reply.mp3",
    tts_voice="nova",
)
print("=== Variant B: Text + TTS ===")
print_result(result_b)

### Variant C: Full voice pipeline (audio → STT → Chat → TTS)

The complete flow. Requires an audio file (e.g. a 5-second recording). If you don't have one, use Variant A or B above.

To record: use your phone's voice recorder, transfer the `.mp3`/`.m4a` file to the `day-1` folder, and set the path below.

In [ ]:
# Variant C: full voice pipeline (uncomment and set your audio file path)
# result_c = run_assistant(
#     audio_path="my_question.mp3",
#     tts_path="voice_reply.mp3",
#     tts_voice="alloy",
# )
# print("=== Variant C: Full voice pipeline ===")
# print_result(result_c)
print("Variant C: Uncomment above and set audio_path to a real .mp3/.wav file.")

### Scenario: IT Help Desk Assistant

**Use case:** An employee speaks their IT issue; the assistant diagnoses and responds with resolution steps. Uses a support-specific system prompt.

In [ ]:
IT_HELPDESK_PROMPT = (
    "You are an IT help desk assistant for an enterprise company. "
    "Diagnose the issue based on the user's description. "
    "Give a numbered list of resolution steps (max 5 steps). "
    "If the issue may require escalation, say so at the end. "
    "Be concise and use simple language."
)

helpdesk_queries = [
    "My laptop won't connect to the VPN. I get a timeout error.",
    "I can't access the shared drive. It says permission denied.",
    "My email is not syncing on my phone since this morning.",
]

for query in helpdesk_queries:
    print(f"\nEmployee: {query}")
    result = run_assistant(text_input=query, system_prompt=IT_HELPDESK_PROMPT)
    print(f"Help Desk: {result['assistant_text']}")
    print(f"  (Chat: {result['timings'].get('chat_seconds', '?')}s)")

### Scenario: Meeting Summary Pipeline

**Use case:** After a meeting recording is transcribed via Block 1 (STT), the pipeline generates a structured summary with 3 sections: Attendees, Key Decisions, Action Items. Uses a meeting-specific system prompt.

In [ ]:
MEETING_SUMMARY_PROMPT = (
    "You summarize meeting transcripts. Output exactly three sections: "
    "## Attendees, ## Key Decisions, ## Action Items. "
    "Use bullet points under each. Include owners and deadlines where mentioned. Be concise."
)

# Simulating a transcript (in production, this comes from Block 1 STT)
demo_transcript = (
    "Alice: Let's discuss the Q4 product launch timeline. Bob, are we on track for November 15? "
    "Bob: Engineering is on track, but QA needs two more weeks. Realistically December 1. "
    "Alice: That pushes marketing by two weeks too. Carol, can you adjust the campaign? "
    "Carol: Yes, I'll update the schedule by Friday. I'll also need the final feature list from Bob by next Wednesday. "
    "Bob: I'll send it by Tuesday. "
    "Alice: Great. Last item — budget. We're approved for the additional testing infrastructure. "
    "Bob: Perfect, I'll set up the staging environment this week."
)

result_meeting = run_assistant(text_input=demo_transcript, system_prompt=MEETING_SUMMARY_PROMPT)
print("=== Meeting Summary ===")
print(result_meeting["assistant_text"])

### Pipeline telemetry

Track per-block timing to identify bottlenecks. In production, log this to your monitoring system.

In [ ]:
def run_with_telemetry(text_input: str, include_tts: bool = False) -> dict:
    """Run pipeline with detailed telemetry per block."""
    import uuid
    request_id = str(uuid.uuid4())[:8]

    result = run_assistant(
        text_input=text_input,
        tts_path=f"telemetry_{request_id}.mp3" if include_tts else None,
    )

    total = sum(result["timings"].values())
    print(f"Request {request_id} — Total: {total:.2f}s")
    for block, elapsed in result["timings"].items():
        pct = (elapsed / total * 100) if total > 0 else 0
        bar = "#" * int(pct / 5)
        print(f"  {block}: {elapsed:.2f}s ({pct:.0f}%) {bar}")

    return result

# Run telemetry
run_with_telemetry("What are three best practices for securing REST APIs?")

## 6. Lab extensions (optional stretch goals)

**These are for participants who finish early.** Skip if you're short on time — the core pipeline (§5) is the primary deliverable.

### Extension 1: Batch mode — process multiple queries

**Use case:** Run 3–5 questions through the assistant in sequence. Collect all answers for review.

In [ ]:
batch_questions = [
    "What is an API gateway?",
    "Explain the difference between REST and GraphQL in two sentences.",
    "What is rate limiting and why is it important?",
]

if API_KEY:
    print("=== Batch mode: 3 queries ===")
    for i, q in enumerate(batch_questions):
        result = run_assistant(text_input=q)
        print(f"\nQ{i+1}: {q}")
        print(f"A{i+1}: {result['assistant_text'][:200]}")
        print(f"  ({result['timings'].get('chat_seconds', '?')}s)")
else:
    print("Set OPENAI_API_KEY to run batch mode.")

### Extension 2: Custom system prompt

**Challenge:** Write your own system prompt for a domain-specific assistant (e.g. HR policy bot, finance calculator, code reviewer). Test it with 2–3 relevant questions.

In [ ]:
# Example: HR Policy Assistant
HR_PROMPT = (
    "You are an HR policy assistant for a mid-size tech company. "
    "Answer employee questions about company policies based on standard best practices. "
    "Keep answers under 3 sentences. If unsure, recommend contacting HR directly."
)

hr_questions = [
    "How many vacation days do new employees get?",
    "What is the policy on remote work?",
]

if API_KEY:
    print("=== Custom System Prompt: HR Assistant ===")
    for q in hr_questions:
        result = run_assistant(text_input=q, system_prompt=HR_PROMPT)
        print(f"\nQ: {q}")
        print(f"A: {result['assistant_text'][:200]}")
else:
    print("Set OPENAI_API_KEY to test custom system prompts.")

### Extension 3: Cost estimation

**Challenge:** Track and estimate the cost of your pipeline runs. Uses the response metadata from Block 2.

In [ ]:
def run_with_cost(text_input: str, price_per_1k_prompt: float = 0.00015, price_per_1k_completion: float = 0.0006) -> dict:
    """Run pipeline and estimate cost."""
    if not API_KEY:
        return {"cost_usd": 0, "tokens": {}}
    content, meta = chat_response_with_metadata(text_input)
    pt = meta.get("prompt_tokens") or 0
    ct = meta.get("completion_tokens") or 0
    cost = (pt / 1000) * price_per_1k_prompt + (ct / 1000) * price_per_1k_completion
    return {
        "answer": content[:100] if content else None,
        "prompt_tokens": pt,
        "completion_tokens": ct,
        "total_tokens": pt + ct,
        "estimated_cost_usd": round(cost, 6),
    }

if API_KEY:
    cost_result = run_with_cost("What is the capital of Japan?")
    print("Cost analysis:")
    print(json.dumps(cost_result, indent=2))
    print(f"\nAt this rate, 1000 similar queries would cost ~${cost_result['estimated_cost_usd'] * 1000:.4f}")
else:
    print("Set OPENAI_API_KEY to estimate costs.")

## 7. Success criteria

**Your pipeline is complete when:**

1. Text in → text out from LLM works for at least one example (Variant A).
2. You can explain which code function implements each pipeline block (STT=Block 1, Chat=Block 2, TTS=Block 3).
3. Optionally: TTS file created (e.g. `assistant_reply.mp3`) and playable locally (Variant B).
4. Optionally: Full voice pipeline works with a real audio file (Variant C).
5. You tried at least two different system prompts and observed how the response changed.
6. Per-block timing is logged so you can identify the slowest block.
7. Error recovery works: if TTS fails, the text answer is still returned.
8. You can describe, in your own words, how the pipeline diagram maps to the code.

### Self-validation

Run this cell to verify your implementations are defined and callable.

In [ ]:
def validate_lab():
    """Self-check: verify all pipeline functions are defined and return expected types."""
    checks = []

    # Check functions exist
    for fn_name in ["speech_to_text", "chat_response", "text_to_speech", "run_assistant"]:
        exists = fn_name in dir() or fn_name in globals()
        checks.append({"check": f"{fn_name} defined", "pass": exists})

    # Check run_assistant returns expected keys
    result = run_assistant(text_input="Test query — validate lab")
    expected_keys = {"user_text", "assistant_text", "tts_path", "timings", "errors"}
    has_keys = expected_keys.issubset(set(result.keys()))
    checks.append({"check": "run_assistant returns expected keys", "pass": has_keys})

    # Check user_text is populated
    checks.append({"check": "user_text is populated", "pass": bool(result.get("user_text"))})

    # Check assistant_text is populated (requires API key)
    has_answer = bool(result.get("assistant_text")) and "[Set OPENAI" not in str(result.get("assistant_text", ""))
    checks.append({"check": "assistant_text is populated (needs API key)", "pass": has_answer})

    # Check TTS files if they exist
    tts_files = list(Path(".").glob("*.mp3"))
    checks.append({"check": f"TTS files generated ({len(tts_files)} found)", "pass": len(tts_files) > 0})

    # Print results
    print("Lab validation:")
    all_pass = True
    for c in checks:
        status = "PASS" if c["pass"] else "FAIL"
        if not c["pass"]:
            all_pass = False
        print(f"  [{status}] {c['check']}")
    print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME CHECKS FAILED (see above)'}")

validate_lab()